# fMRI Preprocessing with fMRIPrep: A Beginner's Tutorial

**Last updated:** March 2026

---

## What is fMRI Preprocessing?

Raw fMRI data is noisy. Before you can analyze brain activity, you need to correct for a variety of artifacts introduced by the scanner, the participant's body, and physics. **Preprocessing** is the pipeline of steps that transforms raw scanner output into clean, analysis-ready data.

**fMRIPrep** is a robust, community-maintained tool that automates this pipeline. It is:
- **Opinionated** — it makes sensible defaults so you don't have to
- **Transparent** — it generates detailed HTML reports for quality checking
- **Reproducible** — containerized (Docker/Singularity) so it runs the same on any machine

### Full pipeline overview

```
Raw DICOM (from scanner)
        │
        ▼  Part 1
   dcm2niix + dcm2bids   ← convert & organize into BIDS format
        │
        ▼  Parts 2–6
   fMRIPrep              ← automated preprocessing
        │
        ▼  Parts 7–8
   QC                    ← quality check outputs
        │
        ▼
   Analysis-ready data   → pass confounds to your GLM design matrix
```

### What fMRIPrep corrects

| Step | What it corrects | Why it matters |
|------|-----------------|----------------|
| Slice timing correction | Each slice is acquired at a slightly different time | Aligns all slices to the same time point |
| Head motion correction | Participant moved during scanning | Realigns all volumes to a reference |
| Susceptibility distortion correction | Field inhomogeneities warp EPI images | Unwarps the image to match anatomy |
| Co-registration | Functional and structural images are in different spaces | Aligns fMRI to T1w for anatomical reference |
| Normalization | Each person's brain is shaped differently | Warps brains to a standard template (MNI) |
| Confound estimation | Physiological noise, motion parameters | Provides regressors to include in your GLM |

---

## Part 1: From Raw Scanner Data to BIDS

### What comes off the scanner?

MRI scanners output data in **DICOM** format — a collection of hundreds (sometimes thousands) of small `.dcm` files, one per slice per volume. Before fMRIPrep can do anything, you need to:

1. Convert DICOM → NIfTI (`.nii.gz`), one file per scan sequence
2. Organize those NIfTI files into the **BIDS** folder structure
3. Generate the required sidecar JSON metadata files

The two tools that do this are:
- **`dcm2niix`** — converts DICOM files to NIfTI + JSON sidecars
- **`dcm2bids`** — wraps dcm2niix and renames/organizes output into BIDS using a config file you write

---

### Step 1.1 — Install the tools

In [ ]:
# dcm2niix: core DICOM → NIfTI converter
#   conda install -c conda-forge dcm2niix
# Or download binary: https://github.com/rordenlab/dcm2niix/releases

# dcm2bids: BIDS organizer wrapper around dcm2niix
#   pip install dcm2bids
#   conda install -c conda-forge dcm2bids

import subprocess

for tool in ["dcm2niix", "dcm2bids"]:
    result = subprocess.run([tool, "--version"], capture_output=True, text=True)
    output = (result.stdout or result.stderr).strip()
    print(f"{tool}: {output[:80]}")

### Step 1.2 — Explore your raw DICOM folder

Your DICOM folder from the scanner typically looks like this:

```
dicom/
└── sub-01/
    ├── 001_localizer/          ← ignore this
    ├── 002_T1w_MPRAGE/         ← structural T1w scan
    ├── 003_task-rest_bold/     ← resting-state fMRI
    ├── 004_task-stroop_bold/   ← task fMRI run 1
    ├── 005_task-stroop_bold/   ← task fMRI run 2
    ├── 006_fmap_AP/            ← spin-echo fieldmap (AP direction)
    └── 007_fmap_PA/            ← spin-echo fieldmap (PA direction)
```

Run `dcm2niix` on **one sequence first** to preview the JSON sidecar. This tells you the exact `SeriesDescription` and other metadata fields you'll need when writing the dcm2bids config file.

In [ ]:
import subprocess, json, os

# Preview a single sequence — change this path to any one DICOM folder
dicom_seq_dir = "/path/to/dicom/sub-01/003_task-rest_bold"
preview_out   = "/tmp/dcm2niix_preview"
os.makedirs(preview_out, exist_ok=True)

result = subprocess.run([
    "dcm2niix",
    "-o", preview_out,   # output directory
    "-f", "%s_%p",       # filename: SeriesNumber_ProtocolName
    "-z", "y",           # compress to .nii.gz
    "-b", "y",           # create BIDS sidecar JSON
    dicom_seq_dir
], capture_output=True, text=True)

print(result.stdout)

# Print the JSON sidecar — we need these field values for the config file
for fname in os.listdir(preview_out):
    if fname.endswith(".json"):
        with open(os.path.join(preview_out, fname)) as f:
            data = json.load(f)
        print(f"\n--- {fname} ---")
        for key in ["SeriesDescription", "ProtocolName", "ImageType",
                    "RepetitionTime", "EchoTime", "PhaseEncodingDirection",
                    "SliceTiming", "MultibandAccelerationFactor"]:
            if key in data:
                print(f"  {key}: {data[key]}")

### Step 1.3 — Write the dcm2bids configuration file

The **dcm2bids config file** is a JSON file that tells dcm2bids how to match each DICOM sequence to a BIDS filename. You match sequences using fields from the JSON sidecar you just previewed — most commonly `SeriesDescription`.

Save this as `dcm2bids_config.json` in your project root and edit it to match your sequences:

In [ ]:
import json

config = {
    "descriptions": [

        # ── Structural T1w ────────────────────────────────────────────────────
        {
            "datatype": "anat",
            "suffix": "T1w",
            "criteria": {
                # Use the exact SeriesDescription string from your JSON preview above
                "SeriesDescription": "T1w_MPRAGE"
            }
        },

        # ── Resting-state fMRI ────────────────────────────────────────────────
        {
            "datatype": "func",
            "suffix": "bold",
            "custom_entities": "task-rest",
            "criteria": {
                "SeriesDescription": "task-rest_bold"
            },
            # sidecar_changes: fields to add/overwrite in the JSON sidecar
            # TaskName is required by BIDS but dcm2niix doesn't add it automatically
            "sidecar_changes": {
                "TaskName": "rest"
            }
        },

        # ── Task fMRI (two runs) ──────────────────────────────────────────────
        # If multiple sequences match the same criteria, dcm2bids auto-assigns
        # run-01, run-02, etc. in acquisition order
        {
            "datatype": "func",
            "suffix": "bold",
            "custom_entities": "task-stroop",
            "criteria": {
                "SeriesDescription": "task-stroop_bold"
            },
            "sidecar_changes": {
                "TaskName": "stroop"
            }
        },

        # ── Spin-echo fieldmaps (blip-up / blip-down) ─────────────────────────
        {
            "datatype": "fmap",
            "suffix": "epi",
            "custom_entities": "dir-AP",
            "criteria": {
                "SeriesDescription": "fmap_AP",
                # PhaseEncodingDirection: 'j-' = Anterior→Posterior on most Siemens scanners
                # Check your preview JSON — this differs between vendors/protocols
                "PhaseEncodingDirection": "j-"
            },
            # IntendedFor: tells fMRIPrep which BOLD files to apply this fieldmap to
            "intended_for": {
                "suffixes": ["bold"]   # applies to all BOLD runs in the same session
            }
        },
        {
            "datatype": "fmap",
            "suffix": "epi",
            "custom_entities": "dir-PA",
            "criteria": {
                "SeriesDescription": "fmap_PA",
                "PhaseEncodingDirection": "j"   # 'j' = Posterior→Anterior
            },
            "intended_for": {
                "suffixes": ["bold"]
            }
        }
    ]
}

config_path = "/path/to/my_study/dcm2bids_config.json"  # <-- save here
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"Config saved to {config_path}")

> **Matching tips:**
> - Use `"*rest*"` for wildcard matching on `SeriesDescription`
> - Add a second criterion (e.g., `PhaseEncodingDirection`) if two sequences share the same description
> - Repeat the same description block for multiple runs — dcm2bids assigns `run-01`, `run-02` automatically

### Step 1.4 — Create the BIDS scaffold

The scaffold command creates the required top-level BIDS files so you don't have to make them by hand:

In [ ]:
import subprocess

bids_dir = "/path/to/my_study"  # <-- your BIDS output folder (will be created)

# Run once per project
result = subprocess.run(
    ["dcm2bids_scaffold", "-o", bids_dir],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

# This creates:
#   my_study/
#   ├── CHANGES
#   ├── README
#   ├── dataset_description.json   ← fill in below
#   ├── participants.json
#   └── participants.tsv

In [ ]:
import json

# Fill in dataset_description.json — required by BIDS
bids_dir = "/path/to/my_study"

dataset_desc = {
    "Name": "My fMRI Study",          # <-- your study name
    "BIDSVersion": "1.9.0",
    "DatasetType": "raw",
    "Authors": ["Firstname Lastname"],  # <-- your name
    "Funding": ["NIH R01-XXXXXX"]       # optional
}

with open(f"{bids_dir}/dataset_description.json", "w") as f:
    json.dump(dataset_desc, f, indent=4)

print("dataset_description.json written.")

### Step 1.5 — Run dcm2bids for each subject

In [ ]:
import subprocess

# ── Edit these ────────────────────────────────────────────────────────────────
RAW_DICOM_DIR = "/path/to/dicom"         # root folder of all subjects' DICOMs
BIDS_DIR      = "/path/to/my_study"      # BIDS output folder
CONFIG_FILE   = "/path/to/my_study/dcm2bids_config.json"
# ─────────────────────────────────────────────────────────────────────────────

def run_dcm2bids(subject_id, dicom_path):
    """
    Convert one subject's DICOMs to BIDS.
    subject_id  : BIDS subject label, e.g. '01'  (do NOT include 'sub-')
    dicom_path  : absolute path to this subject's DICOM folder
    """
    cmd = [
        "dcm2bids",
        "-d", dicom_path,   # DICOM input directory
        "-p", subject_id,   # participant/subject label
        "-c", CONFIG_FILE,  # config file
        "-o", BIDS_DIR,     # BIDS output directory
        "--force_dcm2niix", # always re-run dcm2niix (safe default)
        "--clobber",        # overwrite existing output files
    ]
    print(f"Running dcm2bids for sub-{subject_id}...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-2000:])  # last 2000 chars to avoid flooding output
    if result.returncode != 0:
        print("ERROR:", result.stderr[-1000:])
    return result.returncode

# Test with one subject first
# run_dcm2bids("01", f"{RAW_DICOM_DIR}/sub-01")

In [ ]:
import os

# Batch convert all subjects
# Map: BIDS subject label -> subfolder name inside RAW_DICOM_DIR
subjects = {
    "01": "sub-01",
    "02": "sub-02",
    "03": "sub-03",
}

failed = []
for subj_label, dicom_subdir in subjects.items():
    dicom_path = os.path.join(RAW_DICOM_DIR, dicom_subdir)
    if not os.path.isdir(dicom_path):
        print(f"  sub-{subj_label}: DICOM folder not found at {dicom_path}")
        failed.append(subj_label)
        continue
    # Uncomment to run:
    # rc = run_dcm2bids(subj_label, dicom_path)
    # if rc != 0:
    #     failed.append(subj_label)

if failed:
    print(f"\nFailed subjects: {failed}")
else:
    print("All subjects converted (or queued).")

### Step 1.6 — Check for unmatched files

dcm2bids places files it **could not match** to any config description into:
```
my_study/tmp_dcm2bids/sub-01/
```
Always check this folder after conversion. Unmatched files mean your config criteria didn't match a sequence.

Common causes:
- Typo in `SeriesDescription` — copy-paste it exactly from the JSON preview
- Scout/localizer scans (safe to ignore)
- Derived/MoCo series from Siemens (safe to ignore — add `"ImageType"` criterion to exclude them)

In [ ]:
import os

bids_dir = "/path/to/my_study"
tmp_dir  = os.path.join(bids_dir, "tmp_dcm2bids")

if not os.path.isdir(tmp_dir):
    print("No tmp_dcm2bids folder found — all files matched!")
else:
    for subj in sorted(os.listdir(tmp_dir)):
        subj_path = os.path.join(tmp_dir, subj)
        if not os.path.isdir(subj_path):
            continue
        nii_files = [f for f in os.listdir(subj_path) if f.endswith(".nii.gz")]
        if nii_files:
            print(f"\n{subj}: {len(nii_files)} unmatched NIfTI(s) — review your config")
            for fname in sorted(nii_files):
                # Load sidecar to show SeriesDescription
                json_path = os.path.join(subj_path, fname.replace(".nii.gz", ".json"))
                desc = ""
                if os.path.exists(json_path):
                    import json
                    with open(json_path) as f:
                        meta = json.load(f)
                    desc = meta.get("SeriesDescription", "")
                print(f"    {fname}  →  SeriesDescription: '{desc}'")
        else:
            print(f"{subj}: OK (no unmatched files)")

### Step 1.7 — Add participants.tsv

`participants.tsv` stores subject-level demographics. It is optional but strongly recommended for reproducibility.

In [ ]:
import pandas as pd

bids_dir = "/path/to/my_study"

# participant_id must exactly match your sub-XX folder names
participants = pd.DataFrame([
    {"participant_id": "sub-01", "age": 25, "sex": "F", "group": "control"},
    {"participant_id": "sub-02", "age": 31, "sex": "M", "group": "control"},
    {"participant_id": "sub-03", "age": 28, "sex": "F", "group": "patient"},
])

participants.to_csv(f"{bids_dir}/participants.tsv", sep="\t", index=False)
print("participants.tsv:")
print(participants.to_string(index=False))

### Step 1.8 — Validate the BIDS dataset

Before running fMRIPrep, **always validate** your BIDS dataset. The validator catches naming errors, missing metadata, and structural problems that will cause fMRIPrep to fail or silently produce wrong results.

> **Web alternative (no install required):** drag-and-drop your folder at https://bids-standard.github.io/bids-validator/

In [ ]:
import subprocess

bids_dir = "/path/to/my_study"

# pip install bids-validator
result = subprocess.run(
    ["bids-validator", bids_dir, "--verbose"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode == 0:
    print("\nBIDS validation passed! Ready for fMRIPrep.")
else:
    print("\nBIDS validation found issues — fix before running fMRIPrep.")

### Common BIDS validation errors and fixes

| Error | Cause | Fix |
|-------|-------|-----|
| `NOT_INCLUDED` | File exists but isn't recognized | Check filename — likely a typo or missing entity |
| `TASK_NAME_MUST_DEFINE` | `TaskName` missing from BOLD sidecar | Add it to `sidecar_changes` in dcm2bids config |
| `MISSING_SESSION` | Some subjects have `ses-` folders, others don't | Be consistent — all or none |
| `INTENDED_FOR_MUST_EXIST` | Fieldmap's `IntendedFor` points to nonexistent BOLD | Check BOLD filenames match exactly |
| `SLICE_TIMING_NOT_DEFINED` | `SliceTiming` absent from BOLD sidecar | Add it or use `--ignore slicetiming` in fMRIPrep |
| `MISSING_TSV_COLUMN_PARTICIPANTS` | Column in `participants.tsv` lacks JSON descriptor | Create `participants.json` describing each column |

---

## Part 2: The BIDS Structure You Just Created

After successful conversion, your project should look like this:

```
my_study/                              ← BIDS root
├── dataset_description.json
├── participants.tsv
├── dcm2bids_config.json               ← keep for reproducibility
├── sub-01/
│   ├── anat/
│   │   ├── sub-01_T1w.nii.gz
│   │   └── sub-01_T1w.json
│   ├── func/
│   │   ├── sub-01_task-rest_bold.nii.gz
│   │   ├── sub-01_task-rest_bold.json          ← has TR, SliceTiming, etc.
│   │   ├── sub-01_task-stroop_run-01_bold.nii.gz
│   │   ├── sub-01_task-stroop_run-01_bold.json
│   │   ├── sub-01_task-stroop_run-02_bold.nii.gz
│   │   └── sub-01_task-stroop_run-02_bold.json
│   └── fmap/
│       ├── sub-01_dir-AP_epi.nii.gz
│       ├── sub-01_dir-AP_epi.json              ← must contain IntendedFor + PhaseEncodingDirection
│       ├── sub-01_dir-PA_epi.nii.gz
│       └── sub-01_dir-PA_epi.json
├── sub-02/
│   └── ...
└── tmp_dcm2bids/                      ← unmatched files — check, then delete
```

### BIDS naming rules at a glance

| Entity | Format | Example |
|--------|--------|---------|
| Subject | `sub-<label>` | `sub-01`, `sub-pilot` |
| Session (optional) | `ses-<label>` | `ses-baseline`, `ses-1` |
| Task | `task-<label>` | `task-rest`, `task-stroop` |
| Run (optional) | `run-<index>` | `run-01`, `run-02` |
| Direction | `dir-<label>` | `dir-AP`, `dir-PA` |
| Suffix | `_<modality>` | `_T1w`, `_bold`, `_epi` |
| Extension | `.nii.gz` | always compressed NIfTI |

### The BOLD sidecar JSON

Every BOLD `.nii.gz` has a paired `.json` sidecar. Key fields fMRIPrep reads:

```json
{
    "RepetitionTime": 2.0,
    "TaskName": "rest",
    "SliceTiming": [0.0, 0.0667, 0.1333, 0.2, "..."],
    "PhaseEncodingDirection": "j-",
    "EffectiveEchoSpacing": 0.00058,
    "TotalReadoutTime": 0.0348
}
```

> **Always verify** `SliceTiming` and `PhaseEncodingDirection` against your scanner protocol — these are the most commonly incorrect fields.

---

## Part 3: Installation

### Option A: Docker (recommended for local machines)

Docker packages fMRIPrep and all its dependencies into a container. You only need Docker installed.

```bash
# Install Docker Desktop from https://www.docker.com
# Then pull the fmriprep image:
docker pull nipreps/fmriprep:24.1.1
```

### Option B: Singularity/Apptainer (recommended for HPC clusters)

Most HPC clusters don't allow Docker but support Singularity (now called Apptainer).

```bash
# Build singularity image from Docker Hub:
singularity build /path/to/fmriprep-24.1.1.sif docker://nipreps/fmriprep:24.1.1

# Or pull directly:
singularity pull docker://nipreps/fmriprep:24.1.1
```

### Option C: pip install (not recommended for production)

```bash
pip install fmriprep
```

> ⚠️ pip installation requires you to manually manage all dependencies. Use Docker/Singularity whenever possible.

### FreeSurfer License

fMRIPrep uses FreeSurfer for surface reconstruction. You **must** have a FreeSurfer license file.

1. Register for free at https://surfer.nmr.mgh.harvard.edu/registration.html
2. You will receive a `license.txt` file by email
3. Save it somewhere accessible (e.g., `~/freesurfer/license.txt`)

---

## Part 4: Running fMRIPrep

### Basic command structure

```bash
fmriprep <bids_dir> <output_dir> participant \
    --participant-label <label> \
    --fs-license-file <path/to/license.txt> \
    [OPTIONS]
```

The three positional arguments are always:
1. `<bids_dir>` — path to your BIDS dataset
2. `<output_dir>` — where fMRIPrep saves results
3. `participant` — analysis level (always `participant` for subject-level preprocessing)

### Full example with Docker

```bash
docker run --rm \
    -v /path/to/my_study:/data:ro \
    -v /path/to/output:/out \
    -v /path/to/freesurfer/license.txt:/opt/freesurfer/license.txt:ro \
    -v /path/to/work_dir:/work \
    nipreps/fmriprep:24.1.1 \
    /data /out participant \
    --participant-label 01 \
    --fs-license-file /opt/freesurfer/license.txt \
    --output-spaces MNI152NLin2009cAsym:res-2 T1w \
    --work-dir /work \
    --nprocs 8 \
    --mem_mb 16000
```

> **Note on `-v` flags:** Docker uses volume mounts to give the container access to your files.  
> Format: `-v /local/path:/container/path:ro` (`:ro` = read-only, safe for inputs)

### Full example with Singularity (for HPC)

```bash
singularity run --cleanenv \
    -B /path/to/my_study:/data:ro \
    -B /path/to/output:/out \
    -B /path/to/work_dir:/work \
    /path/to/fmriprep-24.1.1.sif \
    /data /out participant \
    --participant-label 01 \
    --fs-license-file /path/to/freesurfer/license.txt \
    --output-spaces MNI152NLin2009cAsym:res-2 T1w \
    --work-dir /work \
    --nprocs 8 \
    --mem_mb 16000
```

> **Note on `-B` flags:** Singularity uses `-B` instead of Docker's `-v` for bind mounts.

---

## Part 5: Key Parameters Explained

### Subject selection

| Parameter | Example | Description |
|-----------|---------|-------------|
| `--participant-label` | `01 02 03` | Which subjects to process. Omit to run all. Do NOT include `sub-`. |
| `--bids-filter-file` | `filter.json` | JSON file to select specific sessions, tasks, or runs |

### Output spaces

`--output-spaces` controls what coordinate spaces your preprocessed data is resampled into:

| Space | Description | Use case |
|-------|-------------|----------|
| `MNI152NLin2009cAsym` | Standard MNI template | Group-level analyses |
| `MNI152NLin2009cAsym:res-2` | MNI at 2mm isotropic | Saves disk space, common default |
| `T1w` | Subject's native anatomical space | ROI analyses using subject's own anatomy |
| `fsnative` | FreeSurfer native surface | Surface-based analyses |
| `fsaverage` | FreeSurfer average surface | Surface-based group analyses |
| `fsaverage5` | Downsampled fsaverage | Faster surface analyses |

You can request multiple spaces at once:
```bash
--output-spaces MNI152NLin2009cAsym:res-2 T1w fsnative
```

### FreeSurfer surface reconstruction

| Parameter | Description |
|-----------|-------------|
| *(default)* | Runs full FreeSurfer recon-all — very slow (6–12 hours/subject) |
| `--fs-no-reconall` | **Skips** FreeSurfer — much faster, volume-only outputs |
| `--anat-derivatives /path` | Use pre-computed FreeSurfer outputs — saves time on reruns |

> **Beginner tip:** Use `--fs-no-reconall` for your first test run. Add surface reconstruction later.

### Fieldmap / distortion correction

fMRIPrep automatically detects fieldmaps in your BIDS dataset:

| Fieldmap type | BIDS files needed | Notes |
|---------------|-------------------|-------|
| Phase-difference | `fmap/*_phasediff.nii.gz` + `*_magnitude1.nii.gz` | Classic Siemens |
| Spin Echo (blip-up/down) | Two `fmap/*_epi.nii.gz` with opposite `PhaseEncodingDirection` | Most robust |
| No fieldmap | None | Uses SyN-SDC (fieldmap-less, less accurate) |

If you have no fieldmap:
```bash
--use-syn-sdc   # fieldmap-less distortion correction
```

### Slice timing correction

```bash
--ignore slicetiming   # skip STC
                        # use if SliceTiming is absent from your JSON sidecar
                        # or if TR < 1s (very fast multiband acquisitions)
```

### Computing resources

| Parameter | Example | Description |
|-----------|---------|-------------|
| `--nprocs` | `8` | Number of CPU cores to use |
| `--mem_mb` | `16000` | RAM limit in MB. fMRIPrep needs at least 8 GB |
| `--omp-nthreads` | `4` | Threads for OpenMP tools (e.g., ANTs). Usually nprocs/2 |

### Work directory

```bash
--work-dir /path/to/work
```
fMRIPrep saves intermediate files here. If the run crashes, it will **resume** from where it left off when you rerun with the same work directory. Delete it only after you're satisfied with results.

---

## Part 6: Practical Example — Task fMRI on a Server Using Docker

This example uses a **task fMRI** (Stroop) dataset on a shared Linux server with Docker. Edit the variables at the top of each script to match your paths.

---

### Step 6.1 — Run one subject (test first!)

Always test with one subject before batching. Run this directly in your terminal.  
Use `tmux new -s fmriprep` first so the job survives SSH disconnections.

```bash
docker run --rm \
    # ── Volume mounts: give the container access to your files ──────────────
    -v /data/my_study:/data:ro \                          # BIDS dataset (read-only)
    -v /data/my_study/derivatives/fmriprep:/out \         # output directory
    -v /data/my_study/work/sub-01:/work \                 # intermediate files (per-subject)
    -v /home/user/freesurfer/license.txt:/opt/freesurfer/license.txt:ro \
    # ── Docker image ────────────────────────────────────────────────────────
    nipreps/fmriprep:24.1.1 \
    # ── Positional arguments ────────────────────────────────────────────────
    /data /out participant \                              # bids_dir  out_dir  level
    # ── Subject ─────────────────────────────────────────────────────────────
    --participant-label 01 \                              # no 'sub-' prefix
    # ── FreeSurfer ──────────────────────────────────────────────────────────
    --fs-license-file /opt/freesurfer/license.txt \
    --fs-no-reconall \                                    # skip surface recon (saves 6–12 hrs)
    # ── Output ──────────────────────────────────────────────────────────────
    --output-spaces MNI152NLin2009cAsym:res-2 \           # MNI space at 2mm isotropic
    --task-id stroop \                                    # only preprocess Stroop BOLD runs
    # ── Resources ───────────────────────────────────────────────────────────
    --work-dir /work \
    --nprocs 8 \                                          # CPU cores
    --mem_mb 24000 \                                      # RAM cap in MB (adjust to your server)
    --omp-nthreads 4                                      # threads for ANTs (≈ nprocs / 2)
```

> **If the run crashes**, fix the error and rerun the exact same command — fMRIPrep will resume from  
> where it left off because the work directory is preserved.

---

### Step 6.2 — Batch script for all subjects

Once sub-01 looks good, save this as `run_fmriprep_all.sh` and run with `bash run_fmriprep_all.sh`.  
Subjects run **sequentially** so RAM usage stays predictable.

```bash
#!/bin/bash
# run_fmriprep_all.sh

# ── Edit these ───────────────────────────────────────────────────────────────
BIDS_DIR="/data/my_study"
OUTPUT_DIR="/data/my_study/derivatives/fmriprep"
FS_LICENSE="/home/user/freesurfer/license.txt"
FMRIPREP_VER="24.1.1"
SUBJECTS=("01" "02" "03" "04" "05")     # subject labels, no 'sub-' prefix
# ─────────────────────────────────────────────────────────────────────────────

mkdir -p "$OUTPUT_DIR"

for SUBJ in "${SUBJECTS[@]}"; do
    echo "========================================"
    echo " Starting fMRIPrep for sub-${SUBJ}"
    echo "========================================"

    WORK_DIR="${BIDS_DIR}/work/sub-${SUBJ}"   # separate work dir per subject
    mkdir -p "$WORK_DIR"

    docker run --rm \
        -v "${BIDS_DIR}:/data:ro" \
        -v "${OUTPUT_DIR}:/out" \
        -v "${WORK_DIR}:/work" \
        -v "${FS_LICENSE}:/opt/freesurfer/license.txt:ro" \
        "nipreps/fmriprep:${FMRIPREP_VER}" \
        /data /out participant \
        --participant-label "${SUBJ}" \
        --fs-license-file /opt/freesurfer/license.txt \
        --fs-no-reconall \
        --output-spaces MNI152NLin2009cAsym:res-2 \
        --task-id stroop \
        --work-dir /work \
        --nprocs 8 \
        --mem_mb 24000 \
        --omp-nthreads 4

    # Report success or failure for each subject
    if [ $? -eq 0 ]; then
        echo "sub-${SUBJ}: DONE"
    else
        echo "sub-${SUBJ}: FAILED — check logs in ${OUTPUT_DIR}/sub-${SUBJ}/log/"
    fi
done

echo "All subjects finished."
```

> **Useful commands while running:**
> ```bash
> docker ps                              # see which container is currently running
> docker stats                           # live CPU / RAM usage
> tail -f /data/my_study/work/sub-01/...  # follow intermediate logs
> ```

## Part 7: Understanding fMRIPrep Outputs

After a successful run, your output directory looks like:

```
derivatives/fmriprep/
├── sub-01.html                          ← QC report — open this in a browser first!
├── logs/
└── sub-01/
    ├── anat/
    │   ├── sub-01_desc-preproc_T1w.nii.gz                          ← skull-stripped T1w
    │   ├── sub-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_T1w.nii.gz
    │   └── sub-01_from-T1w_to-MNI152NLin2009cAsym_mode-image_xfm.h5  ← warp field
    └── func/
        ├── sub-01_task-stroop_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
        │                                ← preprocessed BOLD — your analysis input
        ├── sub-01_task-stroop_run-01_desc-confounds_timeseries.tsv
        │                                ← confound regressors (motion, WM, CSF, FD…)
        └── sub-01_task-stroop_run-01_desc-brain_mask.nii.gz
```

### Key output files

| File | What it contains | When you use it |
|------|-----------------|-----------------|
| `*_desc-preproc_bold.nii.gz` | Preprocessed BOLD in requested space | Input to your 1st-level GLM |
| `*_desc-confounds_timeseries.tsv` | ~100 confound columns (motion params, FD, WM, CSF, aCompCor…) | Select columns → pass to GLM |
| `sub-XX.html` | Visual QC report with carpet plots, FD trace, brain masks | Check for every subject before analysis |
| `*_xfm.h5` | ANTs warp field (T1w ↔ MNI) | Needed only if you apply transforms manually |

> **Check `sub-XX.html` for every subject.** It takes 2 minutes and will catch registration failures, excessive motion, and fieldmap errors before they contaminate your results.

---

## Part 8: Quality Control

### Step 8.1 — Visual QC: the HTML report

Open `derivatives/fmriprep/sub-XX.html` in a browser. Check for:

| Section | What to look for | Red flag |
|---------|-----------------|----------|
| **T1w brain mask** | Mask covers full brain | Cuts into cortex or includes neck |
| **T1w → MNI registration** | Outlines align on the reference overlay | Gross misalignment, especially cerebellum |
| **BOLD brain mask** | Mask covers full brain for every run | Dropout cuts off frontal/temporal regions |
| **BOLD → T1w registration** | Contours match anatomy | EPI-T1 mismatch in frontal/orbitofrontal |
| **Carpet plot** | Mostly flat, no stripes or sudden jumps | Vertical stripes = motion spikes |
| **Framewise displacement** | Most frames < 0.5 mm | Sustained FD > 0.5 mm; large sudden spikes |
| **CompCor components** | aCompCor components capture noise | If first components are motion-like, flag |

> **Reject criteria (common thresholds):**  
> - Mean FD > 0.5 mm → exclude subject  
> - > 20% of volumes with FD > 0.5 mm → exclude subject (or exclude those volumes)

---

### Step 8.2 — Automated motion QC across subjects

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ── Edit these ────────────────────────────────────────────────────────────────
FMRIPREP_DIR = "/data/my_study/derivatives/fmriprep"
SUBJECTS     = ["01", "02", "03", "04", "05"]
TASK         = "stroop"
FD_THRESHOLD = 0.5   # mm — standard motion threshold
# ─────────────────────────────────────────────────────────────────────────────

summary = []

for subj in SUBJECTS:
    # Find confounds file(s) for this subject + task
    func_dir = os.path.join(FMRIPREP_DIR, f"sub-{subj}", "func")
    if not os.path.isdir(func_dir):
        print(f"sub-{subj}: func directory not found")
        continue

    confound_files = [
        f for f in os.listdir(func_dir)
        if f"task-{TASK}" in f and "confounds_timeseries" in f and f.endswith(".tsv")
    ]

    for cfile in sorted(confound_files):
        df = pd.read_csv(os.path.join(func_dir, cfile), sep="\t")
        fd = df["framewise_displacement"].dropna()

        mean_fd      = fd.mean()
        pct_high_fd  = (fd > FD_THRESHOLD).mean() * 100  # % of volumes above threshold
        flag         = "EXCLUDE" if mean_fd > FD_THRESHOLD or pct_high_fd > 20 else "OK"

        summary.append({
            "subject": f"sub-{subj}",
            "run": cfile,
            "mean_FD": round(mean_fd, 3),
            "pct_above_threshold": round(pct_high_fd, 1),
            "flag": flag
        })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

# ── Plot mean FD per subject ──────────────────────────────────────────────────
if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ["red" if f == "EXCLUDE" else "steelblue" for f in summary_df["flag"]]
    ax.bar(range(len(summary_df)), summary_df["mean_FD"], color=colors)
    ax.axhline(FD_THRESHOLD, color="red", linestyle="--", label=f"Threshold ({FD_THRESHOLD} mm)")
    ax.set_xticks(range(len(summary_df)))
    ax.set_xticklabels(summary_df["subject"], rotation=45, ha="right")
    ax.set_ylabel("Mean Framewise Displacement (mm)")
    ax.set_title(f"Motion QC — task-{TASK}")
    ax.legend()
    plt.tight_layout()
    plt.show()

---

## Part 9: About confounds

The confounds TSV has ~100 columns. You don't use all of them. The right choice depends on your analysis type:

**Key columns in the confounds TSV:**

| Column | Description |
|--------|-------------|
| `trans_x`, `trans_y`, `trans_z` | Translation motion parameters (mm) |
| `rot_x`, `rot_y`, `rot_z` | Rotation motion parameters (radians) |
| `framewise_displacement` | FD trace — use for scrubbing |
| `white_matter`, `csf` | Mean signal in WM/CSF masks |
| `a_comp_cor_00`…`a_comp_cor_05` | Top anatomical CompCor components (noise) |
| `global_signal` | Whole-brain mean signal (controversial — use only if justified) |

### For task fMRI: pass confounds to your GLM design matrix

**Never pre-clean the BOLD signal for task fMRI.** Instead, pass the confound columns directly to your GLM. This preserves the correct statistical model and keeps degrees of freedom accounted for properly.

### For resting-state: clean the BOLD signal

For connectivity analyses (seed-based, ICA, etc.), you *do* regress out confounds from the BOLD signal before computing correlations. Use `nilearn.signal.clean()` or `nilearn.image.clean_img()`.

---

---

## Part 10: Common Errors and Final Checklist

### Common fMRIPrep errors and fixes

| Error message | Likely cause | Fix |
|---------------|-------------|-----|
| `FileNotFoundError: license.txt` | FreeSurfer license path is wrong | Check the `-v` mount path; make sure the file exists on the host |
| `BIDS validation errors` | BIDS dataset isn't valid | Run `bids-validator` and fix all errors before running fMRIPrep |
| `No BOLD images found` | Filenames don't match BIDS spec | Check `task-`, `run-` entities and `_bold` suffix |
| `Fieldmap could not be used` | `IntendedFor` is missing or wrong | Add `IntendedFor` to fmap JSON sidecars; rerun BIDS validator |
| `killed` / OOM crash | Not enough RAM | Increase `--mem_mb`; use `--fs-no-reconall`; process fewer subjects at once |
| `Workflow did not execute` | Crash mid-run | Check `<work-dir>/fmriprep_wf/*/crash-*.txt` for the actual error |
| `RuntimeError: no space left` | Work dir or output dir is full | Clear old work dirs or move to a larger disk |
| `SliceTiming not defined` | `SliceTiming` missing from JSON sidecar | Add it to dcm2bids config `sidecar_changes`, or use `--ignore slicetiming` |

### Where to find logs

```
derivatives/fmriprep/sub-01/log/                  ← per-subject fMRIPrep logs
<work-dir>/fmriprep_wf/**/crash-*.txt             ← crash reports (most useful)
<work-dir>/fmriprep_wf/**/command.txt             ← exact commands run at each step
```

---

### Final checklist before starting your analysis

**BIDS + fMRIPrep**
- [ ] BIDS validator passes with 0 errors
- [ ] fMRIPrep completed without errors for all subjects
- [ ] HTML QC reports reviewed for every subject
- [ ] High-motion subjects flagged or excluded (mean FD > 0.5 mm)

**Output files**
- [ ] Preprocessed BOLD files exist for all subjects and runs
- [ ] Confounds TSV files exist alongside each BOLD file
- [ ] Output space matches what your analysis tool expects (e.g., MNI 2mm)

**Analysis setup**
- [ ] Confound columns selected (not the raw 100-column TSV)
- [ ] First row NaN in `framewise_displacement` filled with 0
- [ ] For task fMRI: confounds passed to GLM, **not** used to pre-clean BOLD
- [ ] For resting-state: BOLD signal cleaned before computing connectivity

---

*Tutorial complete. If you run into issues not covered here, check the [fMRIPrep documentation](https://fmriprep.org) and the [NeuroStars community forum](https://neurostars.org).*